In [5]:
from pystac import Item
from affine import Affine
from odc.geo.geobox import GeoBox

from ldn.utils import http_to_s3_url

from odc.stac import load, configure_s3_access

url = 'https://landsatlook.usgs.gov/stac-server/collections/landsat-c2l2-sr/items/LC09_L2SR_073073_20240128_20240130_02_T1_SR'
item = Item.from_file(url)

src_geobox = GeoBox(item.properties["proj:shape"], Affine(*item.properties["proj:transform"]), crs=item.properties["proj:epsg"])

# This is wrong... it's world spanning.
src_geobox.geographic_extent

shape = (5000, 5000)
transform = Affine(0.0003, 0.0, 180.0, 0.0, -0.0003, -18.0)
crs = 'EPSG:4326'

dst_geobox = GeoBox(shape, transform, crs)

configure_s3_access(cloud_defaults=True, requester_pays=True)

# Can't load data in the target geobox
data = load([item], geobox=dst_geobox, chunks={}, bands=["red"], patch_url=http_to_s3_url).compute()
data  # Empty

<xarray.Dataset> Size: 100MB
Dimensions:      (latitude: 5000, longitude: 5000, time: 1)
Coordinates:
  * latitude     (latitude) float64 40kB -18.0 -18.0 -18.0 ... -19.5 -19.5 -19.5
  * longitude    (longitude) float64 40kB 180.0 180.0 180.0 ... 181.5 181.5
    spatial_ref  int32 4B 4326
  * time         (time) datetime64[ns] 8B 2024-01-28T22:01:06.923200
Data variables:
    red          (time, latitude, longitude) float32 100MB nan nan ... nan nan

In [8]:
data.red.max()

<xarray.DataArray 'red' ()> Size: 8B
array(nan)
Coordinates:
    spatial_ref  int32 4B 4326

In [20]:
# item.properties["proj:shape"]
item.properties["proj:epsg"]

32601

In [ ]:
src_geobox.geographic_extent.explore()

In [ ]:
src_geobox.geographic_extent.boundingbox

In [ ]:
data = load([item], chunks={}, crs="utm", bands=["red"]).compute()
data

In [ ]:
f